In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.chunking import get_tokenizer, json_to_df_new
from src.class_weights import calculate_pos_weights

print("Imports successful!")

Imports successful!


In [2]:
tokenizer = get_tokenizer()

train_chunks = json_to_df_new(
    project_root / "data/train.json",
    tokenizer,
    max_tokens=450,
    overlap=75,
    strategy="sliding"
)

print("Training chunks:", len(train_chunks))
print(train_chunks["label_status"].value_counts())

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (11248 > 512). Running this sequence through the model will result in indexing errors


Training chunks: 11921
label_status
negative    7240
positive    4466
partial      215
Name: count, dtype: int64


In [3]:
categories = [
    "Cap On Liability",
    "Termination For Convenience",
    "Governing Law",
    "Anti-Assignment",
    "Change Of Control",
    "Renewal Term",
    "Exclusivity",
    "Non-Compete",
    "Notice Period To Terminate Renewal",
    "Liquidated Damages"
]

categories = [category.casefold() for category in categories]

print("Categories:", len(categories))

Categories: 10


In [4]:
from src.chunking import prepare_training_chunks

training_chunks = prepare_training_chunks(
    train_chunks,
    neg_ratio=3,
    seed=42
)

y_train = np.array([
    [int(category in chunk_categories) for category in categories]
    for chunk_categories in training_chunks["categories"]
])

positive_counts = y_train.sum(axis=0)
negative_counts = len(y_train) - positive_counts

weights = calculate_pos_weights(y_train)

results = pd.DataFrame({
    "category": categories,
    "positive_chunks": positive_counts,
    "negative_chunks": negative_counts,
    "pos_weight": weights
})

results = results.sort_values("pos_weight", ascending=False)

display(results)

,category,positive_chunks,negative_chunks,pos_weight
9,liquidated damages,79,11627,100.000000
8,notice period to terminate renewal,112,11594,100.000000
4,change of control,141,11565,82.021277
7,non-compete,157,11549,73.560510
1,termination for convenience,169,11537,68.266272
5,renewal term,175,11531,65.891429
6,exclusivity,255,11451,44.905882
2,governing law,372,11334,30.467742
3,anti-assignment,416,11290,27.139423
0,cap on liability,440,11266,25.604545


In [5]:
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

results.to_csv(
    results_dir / "class_weights.csv",
    index=False
)

print("Saved results/class_weights.csv")

Saved results/class_weights.csv


In [6]:
print("Maximum chunk length:", train_chunks["num_tokens"].max())
print("Chunks over 450 tokens:", (train_chunks["num_tokens"] > 450).sum())
print("Training rows after sampling:", len(training_chunks))

Maximum chunk length: 450
Chunks over 450 tokens: 0
Training rows after sampling: 11706


In [7]:

# Count chunks that have none of our 10 target categories
target_positive = (y_train.sum(axis=1) > 0)

print("Chunks with target categories:", target_positive.sum())
print("Chunks without target categories:", (~target_positive).sum())

print(
    "Percent without target categories:",
    round((~target_positive).mean() * 100, 2),
    "%"
)


Chunks with target categories: 1870
Chunks without target categories: 9836
Percent without target categories: 84.03 %


In [8]:

# Identify chunks positive for at least one target category
has_target = y_train.sum(axis=1) > 0

positive_chunks = training_chunks.loc[has_target]
negative_chunks = training_chunks.loc[~has_target]

# Keep at most 3 negative chunks per positive chunk
n_neg = min(len(negative_chunks), 3 * len(positive_chunks))

sampled_negatives = negative_chunks.sample(
    n=n_neg,
    random_state=42
)

balanced_chunks = pd.concat([
    positive_chunks,
    sampled_negatives
]).sample(frac=1, random_state=42)

print("Before:", len(training_chunks))
print("After:", len(balanced_chunks))
print("Positive:", len(positive_chunks))
print("Negative sampled:", len(sampled_negatives))


Before: 11706
After: 7480
Positive: 1870
Negative sampled: 5610


In [9]:

# Create labels for the new sampled dataset
y_balanced = np.array([
    [int(category in chunk_categories) for category in categories]
    for chunk_categories in balanced_chunks["categories"]
])

# Calculate weights for both datasets
original_weights = calculate_pos_weights(y_train)
sampled_weights = calculate_pos_weights(y_balanced)

comparison = pd.DataFrame({
    "category": categories,
    "original_weight": original_weights,
    "sampled_weight": sampled_weights
})

comparison = comparison.sort_values(
    "original_weight",
    ascending=False
)

display(comparison.round(2))


,category,original_weight,sampled_weight
9,liquidated damages,100.00,93.68
8,notice period to terminate renewal,100.00,65.79
4,change of control,82.02,52.05
7,non-compete,73.56,46.64
1,termination for convenience,68.27,43.26
5,renewal term,65.89,41.74
6,exclusivity,44.91,28.33
2,governing law,30.47,19.11
3,anti-assignment,27.14,16.98
0,cap on liability,25.60,16.00


In [10]:

from pathlib import Path

results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

comparison.to_csv(
    results_dir / "class_weight_comparison.csv",
    index=False
)

print("Saved class_weight_comparison.csv!")


Saved class_weight_comparison.csv!
